# Sparkle V2 — Fase 3.5c: controles a dimensión igualada

**Enmienda al `Protocolo_Exp2_Proyeccion_Apariencia.md`, fechada 2026-07-29.**

## Por qué existe este notebook

El criterio de éxito pre-registrado (§5) comparaba cada brazo contra **R1 d=128**. La corrida
de 3.5b mostró que esa comparación está **confundida**: `R2 d=16 vs R1 d=128` cambia a la vez
el tipo de proyección y la dimensión. Y el efecto dominante resultó ser la dimensión —dentro
del propio R1, pasar de d=128 a d=32 da +2,35 pts, más que la comparación que "ganó".

El defecto es del protocolo, no de la corrida. Esta enmienda agrega los controles que faltaban.
**Nada de esto toca el Test.**

## Qué se agrega

1. **Comparaciones a d igualado** (`R2 d` vs `R1 d`): ¿la PCA aporta *algo* sobre la aleatoria?
2. **Efecto de la dimensión dentro de R1**: ¿es la capacidad lo que explica todo?
3. **Brazo R0 — geometría sola**, el control que faltaba. Si `R1 d=16` empata con R0, la rama
   de apariencia **no aporta nada** y hay que descartarla.
4. **R0 vs el mejor brazo con apariencia**, con IC bootstrap clusterizado.

Solo el punto 3 requiere cómputo (5 regresiones logísticas). El resto sale de
`oof_predicciones.npz`.

In [ ]:
import sys, os, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

CACHE_32 = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
OUT_DIR  = f'{PROJECT_DIR}/fase3_5_proyeccion'
OOF_NPZ  = f'{OUT_DIR}/oof_predicciones.npz'

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED, UMBRAL, N_FOLDS, N_BOOT = 42, 0.5, 5, 5000
DS = [16, 32, 64, 128]
N_FRAMES = 60
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

assert os.path.exists(OOF_NPZ), f'Falta {OOF_NPZ} (lo genera 3.5b §4)'
z = np.load(OOF_NPZ, allow_pickle=True)
y_oof = z['y']; part_oof = np.array([str(p) for p in z['part']])
ids_oof = np.array([str(c) for c in z['ids']])
print(f'OOF cargado: {len(y_oof)} clips · {len(set(part_oof))} participantes')

## 1. Utilidades — mismas definiciones que 3.5b

In [ ]:
def _balacc(y, ok):
    n0 = (y == 0).sum(); n1 = (y == 1).sum()
    if n0 == 0 or n1 == 0:
        return np.nan
    return 0.5 * (ok[y == 0].sum() / n0 + ok[y == 1].sum() / n1)

def metricas(y, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int)
    triv = max((y == 0).mean(), (y == 1).mean())
    return {'n': int(len(y)), 'trivial': round(float(triv), 4),
            'accuracy': round(float(accuracy_score(y, pred)), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'brecha': round(float(accuracy_score(y, pred) - triv), 4),
            'auc': round(float(roc_auc_score(y, p)), 4)}

def boot_pareado(pa, pb, y=y_oof, part=part_oof, n_boot=N_BOOT, seed=SEED):
    '''IC de BalAcc(pb) − BalAcc(pa), remuestreando PARTICIPANTES con reemplazo.'''
    rng = np.random.default_rng(seed)
    parts = np.unique(part)
    idx = {p: np.where(part == p)[0] for p in parts}
    oka = (pa > UMBRAL).astype(int) == y
    okb = (pb > UMBRAL).astype(int) == y
    difs = np.empty(n_boot)
    for i in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        j = np.concatenate([idx[p] for p in sel]); yj = y[j]
        difs[i] = _balacc(yj, okb[j]) - _balacc(yj, oka[j])
    difs = difs[~np.isnan(difs)]
    lo, hi = np.percentile(difs, [2.5, 97.5])
    return {'delta_balacc': round(float(difs.mean()), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)],
            'cruza_cero': bool(lo <= 0 <= hi)}

P = {k: z[k] for k in z.files if k not in ('y', 'part', 'ids')}
print('Brazos disponibles:', sorted(P))

## 2. Control 1 — ¿aporta la PCA sobre la aleatoria, a d igualado?

Esta es la comparación que el protocolo original debió haber hecho: **misma dimensión, lo
único que cambia es cómo se eligen las direcciones.**

In [ ]:
filas = [{'comparacion': f'R2 d={d}  vs  R1 d={d}', **boot_pareado(P[f'R1_d{d}'], P[f'R2_d{d}'])}
         for d in DS]
df1 = pd.DataFrame(filas)
print(df1.to_string(index=False))
if df1.cruza_cero.all():
    print('\n📌 Ningún d muestra ventaja de la PCA sobre la proyección aleatoria a dimensión '
          'igualada.\n   ⇒ La hipótesis "la proyección aleatoria era el defecto" NO se sostiene.')

## 3. Control 2 — ¿es la dimensión lo que explica todo?

In [ ]:
filas = [{'comparacion': f'R1 d={d}  vs  R1 d=128', **boot_pareado(P['R1_d128'], P[f'R1_d{d}'])}
         for d in DS if d != 128]
df2 = pd.DataFrame(filas)
print(df2.to_string(index=False))
print('\nSi estas diferencias son mayores que las del control 1, el efecto es de CAPACIDAD\n'
      '(menos features de apariencia = menos sobreajuste de la cabeza), no de representación.')

## 4. Control 3 — R0, geometría sola. **El control decisivo.**

Se replica el bucle OOF de 3.5b usando **solo** el resumen geométrico, con los mismos folds,
la misma semilla y la misma cabeza. Si `R1 d=16` no le gana a R0, la rama de apariencia no
aporta nada y hay que descartarla del modelo.

In [ ]:
d32 = np.load(CACHE_32, allow_pickle=True)
FEATS22 = [str(f) for f in d32['feature_names']]
Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])
y3_pool = np.concatenate([d32['y3_tr'], d32['y3_va']])
id_pool = np.array([str(c) for c in np.concatenate([d32['id_tr'], d32['id_va']])])
y_pool = (y3_pool > 0).astype(np.int64)
part_pool = np.array([c[:6] for c in id_pool])

def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X); return np.hstack([X.mean(axis=1), X.std(axis=1)])

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_R0 = np.full(len(y_pool), np.nan)
for fold, (tr, va) in enumerate(sgkf.split(Xg_pool, y_pool, groups=part_pool)):
    assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'   # regla 7
    g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
    Xg_a, kga = clean(Xg_pool[tr], g_med); Xg_b, kgb = clean(Xg_pool[va], g_med)
    gsc = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
    sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
    R_tr, R_ev = resumir(sg(add_deltas(Xg_a))), resumir(sg(add_deltas(Xg_b)))
    sc = StandardScaler().fit(R_tr)
    lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    lr.fit(sc.transform(R_tr), y_pool[tr][kga])
    oof_R0[va[kgb]] = lr.predict_proba(sc.transform(R_ev))[:, 1]
    print(f'fold {fold} listo')

# Alinear R0 al mismo subconjunto de clips que el OOF de 3.5b (que además filtró por apariencia)
pos = {c: i for i, c in enumerate(id_pool)}
sel = np.array([pos[c] for c in ids_oof])
p_R0 = oof_R0[sel]
assert not np.isnan(p_R0).any(), ('Hay clips del OOF de 3.5b sin predicción geométrica. '
                                  'No debería pasar: la limpieza geométrica es la misma.')
print(f'\nR0 alineado sobre los mismos {len(p_R0)} clips.')

## 5. Tabla final y veredicto honesto

In [ ]:
tabla = [{'brazo': 'R0 — geometría sola', 'd': '—', **metricas(y_oof, p_R0)}]
for b in ['R1', 'R2', 'R3', 'R3lin']:
    for d in DS:
        tabla.append({'brazo': b, 'd': d, **metricas(y_oof, P[f'{b}_d{d}'])})
df_t = pd.DataFrame(tabla)
print(df_t.to_string(index=False))
df_t.to_csv(f'{OUT_DIR}/controles_tabla.csv', index=False)

print('\n--- ¿Aporta la apariencia SOBRE la geometría sola? ---')
filas = []
for b in ['R1', 'R2']:
    for d in DS:
        filas.append({'comparacion': f'{b} d={d}  vs  R0 (geom sola)',
                      **boot_pareado(p_R0, P[f'{b}_d{d}'])})
df3 = pd.DataFrame(filas)
print(df3.to_string(index=False))

pd.concat([df1.assign(control='PCA vs aleatoria, d igual'),
           df2.assign(control='dimensión dentro de R1'),
           df3.assign(control='apariencia vs geometría sola')]
          ).to_csv(f'{OUT_DIR}/controles_comparaciones.csv', index=False)

print('\n================ VEREDICTO DE LOS CONTROLES ================')
r0 = metricas(y_oof, p_R0)['balanced_acc']
print(f'R0 (geometría sola): BalAcc {r0:.4f}')
gana_ap = df3[(~df3.cruza_cero) & (df3.delta_balacc > 0)]
if len(gana_ap) == 0:
    print('❌ NINGUNA configuración de apariencia supera a la geometría sola con IC que no '
          'cruce cero.')
    print('   ⇒ La rama de apariencia NO aporta información utilizable con este backbone.')
    print('   ⇒ Consecuencia de producto: se puede ELIMINAR el extractor CNN (767 kB, la parte')
    print('     más cara del pipeline en dispositivo) sin pérdida medible.')
    print('   ⇒ Consecuencia para la tesis: es un resultado negativo fuerte y publicable, no')
    print('     un fracaso — contradice el supuesto del plan de Fase 3.2 (§2) de que la')
    print('     apariencia aportaría +12 pts. Documentar con el mismo detalle (regla 10).')
else:
    print('✅ Configuraciones que superan a la geometría sola:')
    print(gana_ap.to_string(index=False))
    print('\n⚠️  d fue seleccionado sobre OOF: el número no sesgado es el del cierre sobre Test.')

if df1.cruza_cero.all():
    print('\n📌 Sobre la hipótesis original del experimento 2: a dimensión igualada la PCA NO')
    print('   supera a la proyección aleatoria. La hipótesis "la proyección aleatoria era el')
    print('   defecto" queda REFUTADA. El +3,0 pts que la ficha atribuye a la apariencia')
    print('   (§4 de Ficha_Tecnica_Modelos.md) hay que revisarlo a la luz de esto.')

## 6. Qué hacer después

1. Registrar TODO en `Fase3_Registro_de_Resultados.md`, incluida la enmienda al protocolo y el
   motivo (la comparación pre-registrada estaba confundida). Regla 10 y regla 3: el desvío se
   documenta, no se disimula.
2. Si el veredicto es negativo, **no correr 2B** (fine-tuning del backbone) esperando que
   cambie el resultado: la evidencia es que la apariencia no aporta *con ninguna proyección*,
   lo que apunta al backbone o al constructo, no a la proyección. 2B pasa a ser una apuesta
   cara (~1 h 45 + reentrenamiento) con una hipótesis debilitada. Decidirlo explícitamente.
3. Revisar el `+3,0 pts` de aporte de apariencia de la ficha técnica: se midió en Test con un
   único artefacto y un IC i.i.d. sobre clips. Con OOF y 82 participantes puede no sobrevivir.
4. El Test sigue **sin abrirse**. Evaluaciones consumidas por Fase 3.5: 0.